# Hyperparameter Tuning for POS Tag Prediction LSTM

This notebook implements a comprehensive hyperparameter search framework:
1. Learning rate finder/sweep
2. Model size search (embedding, hidden, layers)
3. Weight decay search
4. Early stopping based on validation loss
5. Comprehensive logging of all runs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import json
import os
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import copy

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load saved corpora
corpora_dir = "corpora"

with open(os.path.join(corpora_dir, 'uniform_corpus.json'), 'r', encoding='utf-8') as f:
    uniform_corpus = json.load(f)

with open(os.path.join(corpora_dir, 'zipfian_corpus.json'), 'r', encoding='utf-8') as f:
    zipfian_corpus = json.load(f)

print("Loaded corpora:")
print(f"  Uniform: {uniform_corpus['metadata']['total_sentences']:,} sentences")
print(f"  Zipfian: {zipfian_corpus['metadata']['total_sentences']:,} sentences")

# Extract data
uniform_sentences = uniform_corpus['sentences']
uniform_classes = uniform_corpus['classes']
zipfian_sentences = zipfian_corpus['sentences']
zipfian_classes = zipfian_corpus['classes']


In [ ]:
# Recreate vocabulary and class mappings (same as original notebook)
def clean_word(word):
    """Remove disambiguation suffixes like _1, _2"""
    if '_' in word:
        return word.rsplit('_', 1)[0]
    return word

# Build vocabulary from both corpora
all_words = []
for sent in uniform_sentences + zipfian_sentences:
    words = sent.split()
    all_words.extend([clean_word(w) for w in words])

word_counter = Counter(all_words)
vocab = ['<PAD>', '<UNK>'] + sorted(word_counter.keys())
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

# Build class mappings
all_classes = []
for classes in uniform_classes + zipfian_classes:
    all_classes.extend(classes)

class_counter = Counter(all_classes)
class_to_idx = {cls: idx for idx, cls in enumerate(sorted(class_counter.keys()))}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

print(f"Vocabulary size: {len(vocab)}")
print(f"Number of classes: {len(class_to_idx)}")
print(f"Classes: {sorted(class_to_idx.keys())}")


In [ ]:
# Prepare data: for each sentence, create (n-1 words, last word class) pairs
def prepare_data(sentences, sentence_classes, n=5):
    """Prepare data: take first n-1 words, predict class of nth word"""
    X = []  # Sequences of n-1 words
    y = []  # Class of nth word
    
    for sent, classes in zip(sentences, sentence_classes):
        words = sent.split()
        if len(words) >= n:
            # Take first n-1 words, predict class of nth word
            sequence = words[:n-1]
            target_class = classes[n-1]
            
            # Convert words to indices
            seq_indices = [word_to_idx.get(clean_word(w), word_to_idx['<UNK>']) for w in sequence]
            class_idx = class_to_idx[target_class]
            
            X.append(seq_indices)
            y.append(class_idx)
    
    return np.array(X), np.array(y)

# Prepare data (using n=5, so 4-word sequences)
n_context = 5
X_uni, y_uni = prepare_data(uniform_sentences, uniform_classes, n=n_context)
X_zip, y_zip = prepare_data(zipfian_sentences, zipfian_classes, n=n_context)

print(f"Uniform corpus - Total samples: {len(X_uni):,}")
print(f"Zipfian corpus - Total samples: {len(X_zip):,}")
print(f"Sequence length (n-1): {n_context - 1}")
print(f"Number of classes: {len(class_to_idx)}")


In [ ]:
# Create train/val splits (using uniform corpus for hyperparameter tuning)
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_uni, y_uni, test_size=0.2, random_state=42, stratify=y_uni
)

print(f"Train samples: {len(X_train):,}")
print(f"Val samples: {len(X_val):,}")


In [ ]:
# Dataset class
class WordClassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = WordClassDataset(X_train, y_train)
val_dataset = WordClassDataset(X_val, y_val)


In [ ]:
# Model definition
class WordClassPredictor(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=6, num_layers=2, dropout=0.3):
        super(WordClassPredictor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        lstm_out, _ = self.lstm(embedded)  # (batch_size, seq_len, hidden_dim)
        # Take the last output
        last_hidden = lstm_out[:, -1, :]  # (batch_size, hidden_dim)
        last_hidden = self.dropout(last_hidden)
        output = self.fc(last_hidden)  # (batch_size, num_classes)
        return output


In [ ]:
# Learning Rate Finder
def learning_rate_finder(model, train_loader, val_loader, min_lr=1e-6, max_lr=1e-1, num_steps=100, 
                         beta=0.98, device=device):
    """
    Learning rate finder using exponential range search.
    Returns learning rates and corresponding losses.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Create exponential range of learning rates
    lrs = np.logspace(np.log10(min_lr), np.log10(max_lr), num_steps)
    
    # Initialize model
    for param in model.parameters():
        param.data.fill_(0.01)
    
    losses = []
    best_loss = float('inf')
    
    # Use a small subset for LR finding
    train_iter = iter(train_loader)
    
    for i, lr in enumerate(lrs):
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        # Get a batch
        try:
            batch_X, batch_y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch_X, batch_y = next(train_iter)
        
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Smooth loss
        if i == 0:
            smoothed_loss = loss.item()
        else:
            smoothed_loss = beta * smoothed_loss + (1 - beta) * loss.item()
        
        losses.append(smoothed_loss)
        
        # Stop if loss explodes
        if smoothed_loss > 4 * best_loss:
            break
        
        if smoothed_loss < best_loss:
            best_loss = smoothed_loss
    
    return lrs[:len(losses)], losses

print("✓ Learning rate finder defined")


In [ ]:
# Training function with early stopping and first-epoch batch-level tracking
def train_with_early_stopping(model, train_loader, val_loader, lr, weight_decay=0, 
                              max_epochs=50, patience=5, device=device, verbose=False, 
                              track_first_epoch_batches=False):
    """
    Train model with early stopping based on validation loss.
    If track_first_epoch_batches=True, tracks per-batch metrics during first epoch only.
    Returns training history and best model state.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    train_losses = []
    val_losses = []  # Per-epoch validation losses
    val_accuracies = []
    
    # First epoch batch-level tracking (similar to WordClassPrediction.ipynb)
    first_epoch_batch_train_losses = []
    first_epoch_batch_train_accs = []
    first_epoch_batch_val_losses = []
    first_epoch_batch_val_accs = []
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(max_epochs):
        # Training
        model.train()
        train_loss = 0.0
        batch_count = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            batch_loss = loss.item()
            train_loss += batch_loss
            batch_count += 1
            
            # Track per-batch metrics during first epoch only
            if track_first_epoch_batches and epoch == 0:
                first_epoch_batch_train_losses.append(batch_loss)
                
                # Calculate batch accuracy
                _, predicted = torch.max(outputs.data, 1)
                batch_acc = 100 * (predicted == batch_y).sum().item() / batch_y.size(0)
                first_epoch_batch_train_accs.append(batch_acc)
                
                # Evaluate on validation set after each batch (first epoch only)
                model.eval()
                val_loss_batch = 0.0
                val_correct = 0
                val_total = 0
                
                with torch.no_grad():
                    for val_batch_X, val_batch_y in val_loader:
                        val_batch_X, val_batch_y = val_batch_X.to(device), val_batch_y.to(device)
                        val_outputs = model(val_batch_X)
                        val_loss_b = criterion(val_outputs, val_batch_y)
                        val_loss_batch += val_loss_b.item()
                        
                        _, val_predicted = torch.max(val_outputs.data, 1)
                        val_total += val_batch_y.size(0)
                        val_correct += (val_predicted == val_batch_y).sum().item()
                
                val_loss_batch /= len(val_loader)
                val_acc_batch = 100 * val_correct / val_total
                first_epoch_batch_val_losses.append(val_loss_batch)
                first_epoch_batch_val_accs.append(val_acc_batch)
                
                model.train()  # Switch back to training mode
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Full validation at end of epoch
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = 100 * correct / total
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{max_epochs}] - Train Loss: {train_loss:.4f}, "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    result = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'best_val_loss': best_val_loss,
        'best_val_acc': max(val_accuracies) if val_accuracies else 0,
        'epochs_trained': len(train_losses)
    }
    
    # Add first epoch batch-level metrics if tracked
    if track_first_epoch_batches:
        result['first_epoch_batch_train_losses'] = first_epoch_batch_train_losses
        result['first_epoch_batch_train_accs'] = first_epoch_batch_train_accs
        result['first_epoch_batch_val_losses'] = first_epoch_batch_val_losses
        result['first_epoch_batch_val_accs'] = first_epoch_batch_val_accs
    
    return result

print("✓ Training function with early stopping defined")


In [ ]:
# Step 1: Learning Rate Sweep
print("="*60)
print("STEP 1: LEARNING RATE FINDER")
print("="*60)

# Use a small model for LR finding
test_model = WordClassPredictor(
    vocab_size=len(vocab),
    embedding_dim=32,
    hidden_dim=64,
    num_classes=len(class_to_idx),
    num_layers=1,
    dropout=0.3
)

batch_size = 64
train_loader_lr = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader_lr = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Running learning rate finder...")
lrs, losses = learning_rate_finder(test_model, train_loader_lr, val_loader_lr, 
                                   min_lr=1e-6, max_lr=1e-1, num_steps=200)

# Find optimal LR (steepest descent)
losses_array = np.array(losses)
gradients = np.gradient(losses_array)
# Find LR where loss decreases fastest (steepest negative gradient)
optimal_idx = np.argmin(gradients)
optimal_lr = lrs[optimal_idx]

print(f"\nOptimal learning rate: {optimal_lr:.6f}")
print(f"  (at index {optimal_idx}, loss: {losses[optimal_idx]:.4f})")

# Plot LR finder results
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.semilogx(lrs[:len(losses)], losses)
plt.axvline(optimal_lr, color='r', linestyle='--', label=f'Optimal LR: {optimal_lr:.6f}')
plt.xlabel('Learning Rate')
plt.ylabel('Loss')
plt.title('Learning Rate Finder')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogx(lrs[:len(losses)], gradients)
plt.axvline(optimal_lr, color='r', linestyle='--', label=f'Optimal LR: {optimal_lr:.6f}')
plt.xlabel('Learning Rate')
plt.ylabel('Loss Gradient')
plt.title('Loss Gradient vs Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Use a range around optimal LR for hyperparameter search
lr_candidates = [optimal_lr * 0.5, optimal_lr, optimal_lr * 2.0]
print(f"\nLR candidates for hyperparameter search: {[f'{lr:.6f}' for lr in lr_candidates]}")


In [ ]:
# Step 2: Hyperparameter Search
print("="*60)
print("STEP 2: HYPERPARAMETER SEARCH")
print("="*60)

# Define search space
embedding_dims = [16, 32, 64]
hidden_dims = [32, 64, 128]
num_layers_list = [1, 2]
weight_decays = [1e-5, 1e-4, 1e-3]

# Use LR candidates from LR finder
learning_rates = lr_candidates

print(f"Search space:")
print(f"  Embedding dims: {embedding_dims}")
print(f"  Hidden dims: {hidden_dims}")
print(f"  Layers: {num_layers_list}")
print(f"  Weight decays: {weight_decays}")
print(f"  Learning rates: {[f'{lr:.6f}' for lr in learning_rates]}")

total_combinations = len(embedding_dims) * len(hidden_dims) * len(num_layers_list) * len(weight_decays) * len(learning_rates)
print(f"\nTotal combinations: {total_combinations}")

# Results storage
results = []

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

run_id = 0
for emb_dim, hid_dim, n_layers, wd, lr in product(embedding_dims, hidden_dims, num_layers_list, weight_decays, learning_rates):
    run_id += 1
    
    print(f"\n[{run_id}/{total_combinations}] Testing: emb={emb_dim}, hid={hid_dim}, layers={n_layers}, wd={wd:.0e}, lr={lr:.6f}")
    
    # Create model
    model = WordClassPredictor(
        vocab_size=len(vocab),
        embedding_dim=emb_dim,
        hidden_dim=hid_dim,
        num_classes=len(class_to_idx),
        num_layers=n_layers,
        dropout=0.3
    )
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    
    # Train with early stopping (track first epoch batches for first run only)
    track_first_epoch = (run_id == 1)  # Only track first epoch batches for first run to save time
    history = train_with_early_stopping(
        model, train_loader, val_loader,
        lr=lr, weight_decay=wd,
        max_epochs=50, patience=5,
        device=device, verbose=False,
        track_first_epoch_batches=track_first_epoch
    )
    
    # Store results
    result = {
        'run_id': run_id,
        'embedding_dim': emb_dim,
        'hidden_dim': hid_dim,
        'num_layers': n_layers,
        'weight_decay': wd,
        'learning_rate': lr,
        'num_parameters': num_params,
        'best_val_loss': history['best_val_loss'],
        'best_val_acc': history['best_val_acc'],
        'epochs_trained': history['epochs_trained'],
        'final_train_loss': history['train_losses'][-1] if history['train_losses'] else None,
        'final_val_loss': history['val_losses'][-1] if history['val_losses'] else None,
        'final_val_acc': history['val_accuracies'][-1] if history['val_accuracies'] else None
    }
    
    # Store first epoch batch history separately for first run
    if track_first_epoch:
        if 'first_epoch_batch_train_losses' in history:
            result['first_epoch_batch_train_losses'] = history['first_epoch_batch_train_losses']
            result['first_epoch_batch_train_accs'] = history.get('first_epoch_batch_train_accs', [])
            result['first_epoch_batch_val_losses'] = history.get('first_epoch_batch_val_losses', [])
            result['first_epoch_batch_val_accs'] = history.get('first_epoch_batch_val_accs', [])
    
    results.append(result)
    
    print(f"  → Val Loss: {history['best_val_loss']:.4f}, Val Acc: {history['best_val_acc']:.2f}%, "
          f"Params: {num_params:,}, Epochs: {history['epochs_trained']}")

print(f"\n{'='*60}")
print(f"HYPERPARAMETER SEARCH COMPLETE")
print(f"{'='*60}")


In [ ]:
# Convert results to DataFrame and save
results_df = pd.DataFrame(results)

# Sort by validation loss (best first)
results_df = results_df.sort_values('best_val_loss')

print("Top 10 configurations by validation loss:")
print("="*100)
print(results_df.head(10).to_string(index=False))

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = f"hyperparameter_results_{timestamp}.csv"
results_df.to_csv(results_file, index=False)
print(f"\nResults saved to: {results_file}")


In [ ]:
# Analysis and visualization
print("="*60)
print("ANALYSIS")
print("="*60)

best_config = results_df.iloc[0]
print(f"\nBest configuration:")
print(f"  Embedding dim: {best_config['embedding_dim']}")
print(f"  Hidden dim: {best_config['hidden_dim']}")
print(f"  Layers: {best_config['num_layers']}")
print(f"  Weight decay: {best_config['weight_decay']:.0e}")
print(f"  Learning rate: {best_config['learning_rate']:.6f}")
print(f"  Parameters: {best_config['num_parameters']:,}")
print(f"  Best Val Loss: {best_config['best_val_loss']:.4f}")
print(f"  Best Val Acc: {best_config['best_val_acc']:.2f}%")

# Plot first-epoch batch-level metrics if available (from first run)
first_run_with_batches = None
for r in results:
    if 'first_epoch_batch_train_losses' in r and len(r.get('first_epoch_batch_train_losses', [])) > 0:
        first_run_with_batches = r
        break

if first_run_with_batches is not None:
    batch_train_losses = first_run_with_batches['first_epoch_batch_train_losses']
    batch_train_accs = first_run_with_batches.get('first_epoch_batch_train_accs', [])
    batch_val_losses = first_run_with_batches.get('first_epoch_batch_val_losses', [])
    batch_val_accs = first_run_with_batches.get('first_epoch_batch_val_accs', [])
    
    if len(batch_train_losses) > 0:
        print(f"\n{'='*60}")
        print("FIRST EPOCH BATCH-LEVEL METRICS (First Hyperparameter Run)")
        print(f"{'='*60}")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        
        batches = np.arange(1, len(batch_train_losses) + 1)
        
        # Loss plot
        ax = axes[0, 0]
        ax.plot(batches, batch_train_losses, label='Train Loss', linewidth=2, alpha=0.8, color='blue')
        if batch_val_losses:
            ax.plot(batches, batch_val_losses, label='Val Loss', linewidth=2, alpha=0.8, color='red')
        ax.set_xlabel('Batch Number', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.set_title('First Epoch Loss (Per Batch)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
        
        # Accuracy plot
        ax = axes[0, 1]
        if batch_train_accs:
            ax.plot(batches, batch_train_accs, label='Train Accuracy', linewidth=2, alpha=0.8, color='green')
        if batch_val_accs:
            ax.plot(batches, batch_val_accs, label='Val Accuracy', linewidth=2, alpha=0.8, color='orange')
        ax.set_xlabel('Batch Number', fontsize=11)
        ax.set_ylabel('Accuracy (%)', fontsize=11)
        ax.set_title('First Epoch Accuracy (Per Batch)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0, top=100)
        
        # Loss trend (smoothed)
        ax = axes[1, 0]
        window = min(50, len(batch_train_losses) // 10)
        if window > 1 and len(batch_train_losses) > window:
            train_smooth = pd.Series(batch_train_losses).rolling(window=window, center=True).mean()
            ax.plot(batches, batch_train_losses, 'b-', alpha=0.3, linewidth=0.5, label='Train Loss (raw)')
            ax.plot(batches, train_smooth, 'b-', alpha=0.9, linewidth=2, label=f'Train Loss (MA, window={window})')
            if batch_val_losses:
                val_smooth = pd.Series(batch_val_losses).rolling(window=window, center=True).mean()
                ax.plot(batches, batch_val_losses, 'r-', alpha=0.3, linewidth=0.5, label='Val Loss (raw)')
                ax.plot(batches, val_smooth, 'r-', alpha=0.9, linewidth=2, label=f'Val Loss (MA, window={window})')
        else:
            ax.plot(batches, batch_train_losses, 'b-', alpha=0.7, linewidth=1, label='Train Loss')
            if batch_val_losses:
                ax.plot(batches, batch_val_losses, 'r-', alpha=0.7, linewidth=1, label='Val Loss')
        ax.set_xlabel('Batch Number', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.set_title('Loss Trend (Smoothed)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
        
        # Summary statistics
        ax = axes[1, 1]
        ax.axis('off')
        summary_text = f"First Epoch Summary:\n\n"
        summary_text += f"Number of batches: {len(batch_train_losses)}\n"
        summary_text += f"\nTrain Loss:\n"
        summary_text += f"  Initial: {batch_train_losses[0]:.4f}\n"
        summary_text += f"  Final: {batch_train_losses[-1]:.4f}\n"
        if batch_train_accs:
            summary_text += f"\nTrain Accuracy:\n"
            summary_text += f"  Initial: {batch_train_accs[0]:.2f}%\n"
            summary_text += f"  Final: {batch_train_accs[-1]:.2f}%\n"
        if batch_val_losses:
            summary_text += f"\nVal Loss:\n"
            summary_text += f"  Initial: {batch_val_losses[0]:.4f}\n"
            summary_text += f"  Final: {batch_val_losses[-1]:.4f}\n"
        if batch_val_accs:
            summary_text += f"\nVal Accuracy:\n"
            summary_text += f"  Initial: {batch_val_accs[0]:.2f}%\n"
            summary_text += f"  Final: {batch_val_accs[-1]:.2f}%\n"
        
        # Check for overfitting
        if batch_val_losses and len(batch_val_losses) > 10:
            train_loss_decrease = batch_train_losses[0] - batch_train_losses[-1]
            val_loss_increase = batch_val_losses[-1] - batch_val_losses[0]
            if train_loss_decrease > 0 and val_loss_increase > 0:
                summary_text += f"\n⚠️  Early overfitting detected:\n"
                summary_text += f"  Train loss decreased by {train_loss_decrease:.4f}\n"
                summary_text += f"  Val loss increased by {val_loss_increase:.4f}\n"
        
        ax.text(0.1, 0.5, summary_text, fontsize=11, family='monospace', 
                verticalalignment='center', transform=ax.transAxes)
        
        plt.tight_layout()
        plt.savefig(f'first_epoch_batch_metrics_{timestamp}.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"First epoch batch-level metrics plot saved to: first_epoch_batch_metrics_{timestamp}.png")
        
        # Print summary
        print(f"\nFirst Epoch Summary:")
        print(f"  Number of batches: {len(batch_train_losses)}")
        print(f"  Initial train loss: {batch_train_losses[0]:.4f}")
        print(f"  Final train loss: {batch_train_losses[-1]:.4f}")
        if batch_train_accs:
            print(f"  Initial train acc: {batch_train_accs[0]:.2f}%")
            print(f"  Final train acc: {batch_train_accs[-1]:.2f}%")
        if batch_val_losses:
            print(f"  Initial val loss: {batch_val_losses[0]:.4f}")
            print(f"  Final val loss: {batch_val_losses[-1]:.4f}")
        if batch_val_accs:
            print(f"  Initial val acc: {batch_val_accs[0]:.2f}%")
            print(f"  Final val acc: {batch_val_accs[-1]:.2f}%")
        
        # Check for overfitting
        if batch_val_losses and len(batch_val_losses) > 10:
            train_loss_decrease = batch_train_losses[0] - batch_train_losses[-1]
            val_loss_increase = batch_val_losses[-1] - batch_val_losses[0]
            if train_loss_decrease > 0 and val_loss_increase > 0:
                print(f"\n⚠️  Early overfitting detected:")
                print(f"   Train loss decreased by {train_loss_decrease:.4f}")
                print(f"   Val loss increased by {val_loss_increase:.4f}")

# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Val loss vs embedding dim
ax = axes[0, 0]
for emb_dim in embedding_dims:
    subset = results_df[results_df['embedding_dim'] == emb_dim]
    ax.scatter(subset['num_parameters'], subset['best_val_loss'], label=f'emb={emb_dim}', alpha=0.6)
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Best Val Loss')
ax.set_title('Val Loss vs Model Size (by Embedding Dim)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Val loss vs hidden dim
ax = axes[0, 1]
for hid_dim in hidden_dims:
    subset = results_df[results_df['hidden_dim'] == hid_dim]
    ax.scatter(subset['num_parameters'], subset['best_val_loss'], label=f'hid={hid_dim}', alpha=0.6)
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Best Val Loss')
ax.set_title('Val Loss vs Model Size (by Hidden Dim)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Val loss vs layers
ax = axes[0, 2]
for n_layers in num_layers_list:
    subset = results_df[results_df['num_layers'] == n_layers]
    ax.scatter(subset['num_parameters'], subset['best_val_loss'], label=f'layers={n_layers}', alpha=0.6)
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Best Val Loss')
ax.set_title('Val Loss vs Model Size (by Layers)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Val loss vs weight decay
ax = axes[1, 0]
for wd in weight_decays:
    subset = results_df[results_df['weight_decay'] == wd]
    ax.scatter(subset['num_parameters'], subset['best_val_loss'], label=f'wd={wd:.0e}', alpha=0.6)
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Best Val Loss')
ax.set_title('Val Loss vs Model Size (by Weight Decay)')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Val loss vs learning rate
ax = axes[1, 1]
for lr in learning_rates:
    subset = results_df[results_df['learning_rate'] == lr]
    ax.scatter(subset['num_parameters'], subset['best_val_loss'], label=f'lr={lr:.6f}', alpha=0.6)
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Best Val Loss')
ax.set_title('Val Loss vs Model Size (by Learning Rate)')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Val accuracy vs val loss
ax = axes[1, 2]
scatter = ax.scatter(results_df['best_val_loss'], results_df['best_val_acc'], alpha=0.6, 
                     c=results_df['num_parameters'], cmap='viridis', s=50)
ax.set_xlabel('Best Val Loss')
ax.set_ylabel('Best Val Acc (%)')
ax.set_title('Val Acc vs Val Loss (colored by # params)')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Number of Parameters')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'hyperparameter_analysis_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAnalysis plots saved to: hyperparameter_analysis_{timestamp}.png")


In [ ]:
# Summary statistics
print("="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(f"\nModel size statistics:")
print(f"  Min parameters: {results_df['num_parameters'].min():,}")
print(f"  Max parameters: {results_df['num_parameters'].max():,}")
print(f"  Mean parameters: {results_df['num_parameters'].mean():,.0f}")
print(f"  Median parameters: {results_df['num_parameters'].median():,.0f}")

print(f"\nValidation loss statistics:")
print(f"  Best: {results_df['best_val_loss'].min():.4f}")
print(f"  Worst: {results_df['best_val_loss'].max():.4f}")
print(f"  Mean: {results_df['best_val_loss'].mean():.4f}")
print(f"  Median: {results_df['best_val_loss'].median():.4f}")

print(f"\nValidation accuracy statistics:")
print(f"  Best: {results_df['best_val_acc'].max():.2f}%")
print(f"  Worst: {results_df['best_val_acc'].min():.2f}%")
print(f"  Mean: {results_df['best_val_acc'].mean():.2f}%")
print(f"  Median: {results_df['best_val_acc'].median():.2f}%")

# Find smallest model within 5% of best loss
best_loss = results_df['best_val_loss'].min()
threshold = best_loss * 1.05
small_models = results_df[results_df['best_val_loss'] <= threshold].sort_values('num_parameters')

if len(small_models) > 0:
    smallest_good = small_models.iloc[0]
    print(f"\nSmallest model within 5% of best loss:")
    print(f"  Embedding dim: {smallest_good['embedding_dim']}")
    print(f"  Hidden dim: {smallest_good['hidden_dim']}")
    print(f"  Layers: {smallest_good['num_layers']}")
    print(f"  Weight decay: {smallest_good['weight_decay']:.0e}")
    print(f"  Learning rate: {smallest_good['learning_rate']:.6f}")
    print(f"  Parameters: {smallest_good['num_parameters']:,}")
    print(f"  Best Val Loss: {smallest_good['best_val_loss']:.4f}")
    print(f"  Best Val Acc: {smallest_good['best_val_acc']:.2f}%")
